# GLAMMAR AI — Qwen 8.0 Max (Qwen 2.5 3B Instruct)

Chat and reasoning with an open instruct model.

1. Sign into Google. Choose **Runtime → Change runtime type → T4 GPU** (free GPUs are not guaranteed).
2. Choose **Runtime → Run all**, approve after reviewing, then use the controls at the bottom.

GLAMMAR label note: "Qwen 8.0 Max" is a studio label — this notebook runs the open **Qwen 2.5 3B Instruct**.

Weights download into this temporary session. Save results before it disconnects. This notebook runs interactively in Colab — it is not an API server. Free Colab policies: https://research.google.com/colaboratory/faq.html

In [ ]:
%pip -q install transformers accelerate bitsandbytes sentencepiece ipywidgets
print("Dependencies installed. Continue below.")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0))

import gc, ipywidgets as w
from IPython.display import display, Markdown, clear_output
from google.colab import output, userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
output.enable_custom_widget_manager()

REPO = "Qwen/Qwen2.5-3B-Instruct"
try:
    TOKEN = userdata.get("HF_TOKEN")
except Exception:
    TOKEN = None

tokenizer = AutoTokenizer.from_pretrained(REPO, token=TOKEN)
model = AutoModelForCausalLM.from_pretrained(REPO, token=TOKEN, device_map="auto", torch_dtype=torch.float16, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
print("Model loaded:", REPO)

prompt = w.Textarea(placeholder="Enter your prompt (max 8,000 characters)…", layout=w.Layout(width="100%", height="120px"))
send = w.Button(description="Send", button_style="primary", icon="play")
release = w.Button(description="Release GPU memory", icon="trash")
log = w.Output()

def on_send(_):
    send.disabled = True
    with log:
        try:
            text = prompt.value.strip()
            if not text:
                raise ValueError("Enter a prompt first.")
            if len(text) > 8000:
                raise ValueError("Use 8,000 characters or fewer on a free GPU.")
            encoded = tokenizer.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True, return_tensors="pt").to(model.device)
            with torch.inference_mode():
                generated = model.generate(encoded, attention_mask=torch.ones_like(encoded), max_new_tokens=512, do_sample=True, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
            display(Markdown(tokenizer.decode(generated[0, encoded.shape[-1]:], skip_special_tokens=True)))
        except Exception as e:
            print("Could not complete:", e)
        finally:
            send.disabled = False

def on_release(_):
    global model, tokenizer
    model = None
    tokenizer = None
    gc.collect()
    torch.cuda.empty_cache()
    with log:
        clear_output(wait=True)
        print("GPU memory released. Re-run this cell to load the model again.")

send.on_click(on_send)
release.on_click(on_release)
display(w.VBox([prompt, w.HBox([send, release]), log]))